# 04. Từ dataset gốc đến MongoDB

> Đọc sau `00` (hiểu dataset), `02` (data mining) và `03` (kiểm chứng). Notebook này biến mọi thứ đã đào được thành **dữ liệu nạp thẳng vào MongoDB**, rồi chạy các bộ phát hiện **ngay trên MongoDB** để chứng minh chúng cho cùng kết quả với phân tích bằng pandas.

**Quy trình:**
```
7 PCAP + 7 CSV nhãn (gốc repo)
   └─ src/dad_export.py      → data/mongo/*.ndjson.gz   (9 collection, Extended JSON, ~490 KB)
        └─ src/dad_mongo_load.py → MongoDB (Time Series, TTL, validator, index)
             └─ src/dad_mongo_pipelines.py → D1, D2, D4, kiểm tra vật lý, dải giá trị, điều tra
```

**Chạy lại:** cần MongoDB ≥ 7.0. Đặt biến môi trường `MONGODB_URI` (mặc định `mongodb://127.0.0.1:27017`). Notebook này đã chạy với MongoDB 8.0.4 cục bộ.

In [1]:
import os, sys, json, gzip, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from datetime import timedelta
import numpy as np, pandas as pd
from bson import json_util
from pymongo import MongoClient
import dad_export, dad_mongo_load as loader, dad_mongo_pipelines as mp
import dad_mining as dm
from dad_loader import load_all

RAW = os.path.abspath(os.path.join("..", "..")); CACHE = os.path.abspath(os.path.join("..", "data", "processed"))
OUT = os.path.abspath(os.path.join("..", "data", "mongo"))
URI = os.environ.get("MONGODB_URI", "mongodb://127.0.0.1:27017"); DB = "dad"
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)
client = MongoClient(URI, tz_aware=True, serverSelectionTimeoutMS=5000)
print("MongoDB", client.server_info()["version"])

MongoDB 8.0.4


## 1. Từ file gốc tới collection nào

| Nguồn trong dataset gốc | Collection | Số document | Vai trò khi hệ thống chạy |
|---|---|---|---|
| PUBLISH trong 7 PCAP (payload nhiệt độ, thời điểm, IP, cổng, msgid) + PUBACK (để tính độ trễ) | `telemetry_hist` | 32.288 | **Time Series** (`granularity: minutes`), nguồn cho replayer và phân tích lịch sử. **Không** chứa nhãn |
| *(rỗng, được tạo sẵn)* | `telemetry` | 0 | **Time Series có TTL 7 ngày** (`granularity: seconds` cho replay ×60): dữ liệu live do replayer/ingest ghi |
| CONNECT trong PCAP + thống kê `telemetry_hist` + Table 1 bài báo | `devices` | 16 | hồ sơ cảm biến: loại, dải hợp lệ, baseline ban đầu, cờ `quality` / `rate_limited` |
| IP + Hình 1 bài báo + gói NTP | `inrows` | 4 | hồ sơ tủ làm mát |
| CONNECT / DISCONNECT / PUBLISH / PUBACK theo kết nối | `mqtt_sessions` | 112 | kết nối MQTT mỗi ngày: keepalive, số bản tin, số bản tin không được ACK, p99 PUBACK |
| Toàn bộ 101.583 gói, phân loại | `traffic_daily` | 35 | số gói theo ngày × IP × loại (khớp Table 5/6 bài báo) |
| `label = 1` trong CSV | `gt_labels` | 544 | **chỉ để chấm**, không cho bộ phát hiện đọc |
| Nhãn + quy tắc khoảng hở (notebook 02 mục 4) | `gt_incidents` | 11 | cửa sổ sự cố chuẩn kèm kết quả điều tra (mất/thừa/mức trừ) |
| Kết quả notebook 02–03 | `detector_config` | 1 | ngưỡng D1/D2/D4, dải hợp lệ |
| Lịch tấn công | `demo_scenarios` | 4 | đoạn dữ liệu nên replay cho demo (giờ UTC) |
| *(rỗng, có validator)* | `incidents`, `actions` | 0 | sự cố và hành động khắc phục do hệ thống ghi |

File `.xml` không cần dùng (trùng CSV). Các tầng mạng khác (ACK thuần, NTP, NetBIOS) được tóm tắt trong `traffic_daily`, không nạp từng gói.

In [2]:
cols = dad_export.build(RAW, CACHE)
manifest = dad_export.write(cols, OUT)
manifest

,collection,documents,KB_gzip
0,telemetry_hist,32288,478.9
1,devices,16,1.3
2,inrows,4,0.3
3,mqtt_sessions,112,2.9
4,traffic_daily,35,0.8
5,gt_labels,544,5.9
6,gt_incidents,11,0.7
7,detector_config,1,0.9
8,demo_scenarios,4,0.4


## 2. Từ điển dữ liệu

### 2.1 `telemetry_hist` (và `telemetry`): một document = một bản tin nhiệt độ

In [3]:
print(json_util.dumps(cols["telemetry_hist"][0], indent=1, ensure_ascii=False))

{
 "ts": {
  "$date": "2019-10-20T22:00:02.550Z"
 },
 "meta": {
  "site": "CITIC-DC",
  "inrow": 15,
  "sensor": "TFSU",
  "clientId": "152",
  "ip": "10.6.56.36"
 },
 "value": 15.54,
 "seq": 1,
 "conn": "10.6.56.36:47590",
 "puback_ms": 0.092,
 "src": {
  "day": "mon",
  "frame": 61
 }
}


| Trường | Kiểu | Nguồn | Ý nghĩa |
|---|---|---|---|
| `ts` | date (UTC, ms) | thời điểm gói PUBLISH tới broker (PCAP) | **timeField** của Time Series |
| `meta.site` | string | cố định `CITIC-DC` | để mở rộng nhiều site |
| `meta.inrow` | int | IP nguồn → InRow | 13, 15, 23, 25 |
| `meta.sensor` | string | chữ số cuối `clientId`, ánh xạ theo giá trị đo | TFEU / TFSU / TAS / TAR |
| `meta.clientId` | string | gói CONNECT của cùng (IP, cổng) | 131 … 254 |
| `meta.ip` | string | IP nguồn | |
| `value` | double | payload MQTT (`"12.12"` → 12.12) | nhiệt độ °C |
| `seq` | int | `msgid` MQTT | tăng 1 mỗi bản tin trong một kết nối |
| `conn` | string | `ip:cổng` | định danh kết nối TCP; đổi mỗi ngày. D2 dùng để bỏ qua bản tin đầu của kết nối mới |
| `puback_ms` | double / null | PUBACK cùng `conn` + `msgid` | độ trễ broker; `null` = không được ACK (20 bản tin lúc chuyển ngày) |
| `src.day`, `src.frame` | string, int | tên file + số frame | truy ngược về PCAP / CSV gốc |

**Vì sao `meta` không chứa `conn`:** `metaField` quyết định cách MongoDB gom bucket. `conn` đổi mỗi ngày nên đặt ngoài `meta` để mỗi cảm biến chỉ có một chuỗi bucket liên tục.

**Vì sao không có `label`:** nhãn là đáp án. Để chung với dữ liệu sẽ dễ vô tình dùng làm đặc trưng (đúng lỗi rò nhãn ở notebook 03, mục 2). Nhãn nằm riêng ở `gt_labels`.

### 2.2 `devices`, `inrows`, `mqtt_sessions`

In [4]:
print(json_util.dumps(cols["devices"][0], indent=1, ensure_ascii=False))
pd.DataFrame([{k: d[k] for k in ["_id", "inrow", "sensor", "valid_range_c", "baseline_init"]} | {"mean_dad": d["profile_dad"]["mean"], "sd_dad": d["profile_dad"]["std"], "sd_thật": d["profile_real_paper"]["sd"]} for d in cols["devices"]])

{
 "_id": "131",
 "clientId": "131",
 "inrow": 13,
 "ip": "10.6.56.41",
 "position": 1,
 "sensor": "TFEU",
 "sensor_name": "Unit Entering Fluid Temperature",
 "measures_vi": "nước lạnh đi vào tủ",
 "physical_order": 1,
 "expected_period_s": 300.1,
 "profile_dad": {
  "mean": 11.911,
  "std": 0.358,
  "p01": 11.08,
  "p99": 13.79,
  "min": 9.94,
  "max": 14.12,
  "n": 577
 },
 "profile_real_paper": {
  "mean": 11.94,
  "sd": 0.93,
  "min": 9.4,
  "max": 14.3,
  "median": 11.8
 },
 "valid_range_c": [
  6.1,
  16.3
 ],
 "baseline_init": 12.03,
 "quality": "ok",
 "rate_limited": false
}


,_id,inrow,sensor,valid_range_c,baseline_init,mean_dad,sd_dad,sd_thật
0,131,13,TFEU,"[6.1, 16.3]",12.030,11.911,0.358,0.93
1,132,13,TFSU,"[10.8, 19.8]",15.980,15.865,0.352,0.79
2,133,13,TAS,"[15.9, 23.2]",20.000,20.021,0.322,0.38
3,134,13,TAR,"[23.1, 30.1]",27.400,27.365,0.328,0.24
4,151,15,TFEU,"[6.1, 16.3]",11.785,11.684,0.366,0.98
5,152,15,TFSU,"[10.8, 19.8]",15.490,15.360,0.374,0.89
6,153,15,TAS,"[15.9, 23.2]",20.010,20.004,0.322,0.42
7,154,15,TAR,"[23.1, 30.1]",27.250,27.201,0.326,0.25
8,231,23,TFEU,"[6.1, 16.3]",11.220,11.134,0.365,1.00
9,232,23,TFSU,"[10.8, 19.8]",15.045,14.947,0.361,0.88


- `profile_dad`: thống kê trên dữ liệu **sạch** (InRow 13 chỉ lấy thứ Hai và Chủ nhật). `profile_real_paper`: Table 1 của bài báo (14 ngày đo thật).
- `valid_range_c`: dải hợp lệ cho quy tắc RANGE = dải thật của loại cảm biến ± 2 °C (0 vi phạm trên DAD bình thường).
- `baseline_init`: trung vị ngày thứ Hai; giúp D4 có baseline ngay từ đầu khi chưa đủ 288 bản tin.
- `quality` và `rate_limited`: trạng thái do remediator cập nhật khi xử lý sự cố.

In [5]:
pd.DataFrame(cols["inrows"]).drop(columns=["_id"])

,inrow,ip,site,netbotz,clients,ntp_seen_in_capture,note
0,13,10.6.56.41,CITIC-DC,NETBOTZ 1,"[131, 132, 133, 134]",True,
1,15,10.6.56.36,CITIC-DC,NETBOTZ 1,"[151, 152, 153, 154]",True,
2,23,10.6.56.50,CITIC-DC,NETBOTZ 2,"[231, 232, 233, 234]",True,
3,25,10.6.56.34,CITIC-DC,NETBOTZ 2,"[251, 252, 253, 254]",False,NTP qua card mạng thứ hai (theo bài báo)


In [6]:
ses = pd.DataFrame(cols["mqtt_sessions"])
print("Số kết nối:", len(ses), "| keepalive theo ngày:", ses.groupby("capture_day").keepalive_s.first().to_dict())
ses[["conn", "clientId", "capture_day", "connect_ts", "keepalive_s", "clean_session", "n_publish", "n_unacked", "puback_p99_ms"]].head(6)

Số kết nối: 112 | keepalive theo ngày: {'fri': 18000, 'mon': 300, 'sat': 18000, 'sun': 300, 'thu': 300, 'tue': 18000, 'wed': 300}


,conn,clientId,capture_day,connect_ts,keepalive_s,clean_session,n_publish,n_unacked,puback_p99_ms
0,10.6.56.36:47590,152,mon,2019-10-20 22:00:02.549637+00:00,300,True,288,0,0.558
1,10.6.56.36:47588,151,mon,2019-10-20 22:00:02.549709+00:00,300,True,288,0,0.487
2,10.6.56.36:47592,153,mon,2019-10-20 22:00:02.549773+00:00,300,True,288,0,0.594
3,10.6.56.36:47594,154,mon,2019-10-20 22:00:02.550822+00:00,300,True,288,0,0.703
4,10.6.56.34:40816,253,mon,2019-10-20 22:00:02.568211+00:00,300,True,288,0,0.364
5,10.6.56.34:40818,251,mon,2019-10-20 22:00:02.568251+00:00,300,True,288,0,0.371


`mqtt_sessions` phục vụ phần **hạ tầng** của monitor: phiên nào gửi thiếu hoặc thừa (`n_publish` ≠ 288), phiên nào có bản tin không được ACK, độ trễ broker. Cột `keepalive_s` đổi theo ngày là dấu vết của quy trình sinh dữ liệu (02, mục 1.2): giữ lại để minh hoạ monitor "config drift", **không** dùng phát hiện tấn công.

### 2.3 `traffic_daily`, `gt_labels`, `gt_incidents`, `detector_config`, `demo_scenarios`

In [7]:
td = pd.DataFrame(cols["traffic_daily"])
pd.concat([td[["day", "ip", "role", "packets", "labeled_packets"]], pd.json_normalize(td.by_kind)], axis=1).head(10)

,day,ip,role,packets,labeled_packets,publish,puback,session,tcp_control,ntp,netbios
0,fri,10.6.56.1,broker,4952,0,0,4624,16,67,245,0
1,fri,10.6.56.34,InRow 25,2332,0,1152,0,8,1172,0,0
2,fri,10.6.56.36,InRow 15,2413,0,1152,0,8,1171,82,0
3,fri,10.6.56.41,InRow 13,2433,192,1168,0,4,1180,81,0
4,fri,10.6.56.50,InRow 23,2414,0,1152,0,8,1172,82,0
5,mon,10.6.56.1,broker,4905,0,0,4608,16,32,246,3
6,mon,10.6.56.34,InRow 25,2320,0,1152,0,4,1164,0,0
7,mon,10.6.56.36,InRow 15,2402,0,1152,0,4,1164,82,0
8,mon,10.6.56.41,InRow 13,2402,0,1152,0,4,1164,82,0
9,mon,10.6.56.50,InRow 23,2402,0,1152,0,4,1164,82,0


In [8]:
gi = pd.DataFrame(cols["gt_incidents"])
pd.concat([gi[["_id", "local", "source", "kinds"]], pd.json_normalize(gi.evidence)], axis=1)

,_id,local,source,kinds,lost_msgs,extra_msgs,dropped_slots,drop_c,sensors,labeled_msgs
0,GT-01,Tue 22/10 00:15–00:55,gap_rule,[interception],16,0,0,NaN,4,0
1,GT-02,Tue 22/10 01:25–02:25,gap_rule,[interception],24,0,0,NaN,4,0
2,GT-03,Wed 23/10 04:10–06:15,label,[modification],0,0,17,2.0,4,104
3,GT-04,Thu 24/10 03:05–03:22,label,"[duplication, modification]",0,56,3,2.0,4,56
4,GT-05,Thu 24/10 23:03–00:00,gap_rule,[interception],40,0,0,NaN,4,0
5,GT-06,Fri 25/10 03:25–05:00,gap_rule,[interception],60,0,0,NaN,4,0
6,GT-07,Fri 25/10 06:00–06:25,label,"[duplication, modification]",0,96,6,2.0,4,96
7,GT-08,Fri 25/10 14:27–16:22,label,[modification],0,0,15,2.0,4,96
8,GT-09,Sat 26/10 03:00–05:00,gap_rule,[interception],40,0,0,NaN,4,0
9,GT-10,Sat 26/10 06:00–06:25,label,"[duplication, modification]",0,96,6,2.0,4,96


- `gt_incidents.source`: `label` = cửa sổ dựng từ nhãn CSV; `gap_rule` = cửa sổ chặn gói, tự dựng bằng quy tắc khoảng hở vì dataset không gắn nhãn loại này.
- `kinds` và `evidence` là kết quả **điều tra** (notebook 03, mục 5): dùng để so với những gì hệ thống live tự suy ra.

In [9]:
pd.DataFrame(cols["demo_scenarios"])

,_id,capture_day,start,end,note,expected_detectors
0,warmup,mon,2019-10-20 22:00:00+00:00,2019-10-21 22:00:00+00:00,"nạp trước làm baseline, không cần tua",[]
1,duplication,thu,2019-10-24 00:50:00+00:00,2019-10-24 01:35:00+00:00,chèn bản tin 03:05–03:22 giờ địa phương,[D2]
2,modification,wed,2019-10-23 01:45:00+00:00,2019-10-23 04:35:00+00:00,sửa giá trị 04:10–06:15 giờ địa phương,[D4]
3,interception,fri,2019-10-25 01:15:00+00:00,2019-10-25 03:10:00+00:00,chặn gói 03:25–05:00 giờ địa phương,[D1]


## 3. Nạp vào MongoDB

`dad_mongo_load.load()` tạo collection, validator và index, rồi nạp dữ liệu:

In [10]:
counts = loader.load(URI, DB, OUT, drop=True)
db = client[DB]
info = []
for c in db.list_collections():
    if c["name"].startswith("system."):
        continue                      # collection nội bộ của Time Series (bucket) và view
    o = c.get("options", {})
    info.append({"collection": c["name"], "documents": db[c["name"]].estimated_document_count() if c["type"] == "collection" else db[c["name"]].count_documents({}),
                 "loại": "time series" if "timeseries" in o else c["type"], "TTL (giây)": o.get("expireAfterSeconds"),
                 "validator": "có" if o.get("validator") else ""})
pd.DataFrame(info).sort_values("collection").reset_index(drop=True)

,collection,documents,loại,TTL (giây),validator
0,actions,0,collection,NaN,có
1,demo_scenarios,4,collection,NaN,
2,detector_config,1,collection,NaN,
3,devices,16,collection,NaN,có
4,gt_incidents,11,collection,NaN,
5,gt_labels,544,collection,NaN,
6,incidents,0,collection,NaN,có
7,inrows,4,collection,NaN,
8,mqtt_sessions,112,collection,NaN,
9,telemetry,0,time series,604800.0,


### 3.1 Bẫy 1: TTL xoá sạch dữ liệu lịch sử

`expireAfterSeconds: 604800` (7 ngày) là tính năng khuyến khích trong đề bài. Nhưng dữ liệu DAD là **năm 2019**: nạp vào collection có TTL thì mọi document đều "quá hạn". Thử chép 500 bản tin vào `telemetry` (có TTL):

In [11]:
import time
try:
    client.admin.command({"setParameter": 1, "ttlMonitorSleepSecs": 2})     # chỉ để demo nhanh; mặc định 60 giây
except Exception as e:
    print("Không đổi được ttlMonitorSleepSecs (Atlas không cho):", e)
db.telemetry.delete_many({})
db.telemetry.insert_many(list(db.telemetry_hist.find({"meta.clientId": "131"}, {"_id": 0}).limit(500)))
print("vừa chèn:", db.telemetry.count_documents({}))
t0 = time.time()
while time.time() - t0 < 120 and db.telemetry.count_documents({}) > 0:
    time.sleep(2)
print(f"sau {time.time() - t0:.0f} giây còn:", db.telemetry.count_documents({}), "| telemetry_hist (không TTL) còn:", db.telemetry_hist.count_documents({}))
try:
    client.admin.command({"setParameter": 1, "ttlMonitorSleepSecs": 60})
except Exception:
    pass

vừa chèn: 500


sau 2 giây còn: 0 | telemetry_hist (không TTL) còn: 32288


**Cách xử lý (đã làm trong `dad_mongo_load.py`):** tách hai collection. `telemetry_hist` giữ dữ liệu gốc, **không TTL**. `telemetry` là dữ liệu live, **có TTL**; replayer **dời thời gian về hiện tại** trước khi ghi vào đây (mục 5).

### 3.2 Dữ liệu có toàn vẹn sau khi chuyển sang MongoDB không?

Đối chiếu **từng bản tin** trong MongoDB với bản tin gốc giải mã từ PCAP, ghép theo `(ngày, số frame)`:

In [12]:
import glob
m = pd.DataFrame(list(db.telemetry_hist.find({}, {"_id": 0, "ts": 1, "value": 1, "seq": 1, "meta.clientId": 1, "src": 1})))
m["clientId"] = m.meta.str["clientId"]; m["day"] = m.src.str["day"]; m["frame"] = m.src.str["frame"]
packets0, pubs0 = load_all(RAW, CACHE)
j = pubs0.assign(day=pubs0.day.astype(str)).merge(m, on=["day", "frame"], how="outer", indicator=True)
b = j[j._merge == "both"]
dt_ms = (pd.to_datetime(b.ts_y, utc=True) - b.time_utc).abs().dt.total_seconds() * 1000
lab_csv = pd.concat([pd.read_csv(f, sep=";") for f in sorted(glob.glob(os.path.join(RAW, "*_labels.csv")))])
pd.DataFrame([
    ("số bản tin (gốc / MongoDB)", f"{len(pubs0)} / {len(m)}", len(pubs0) == len(m)),
    ("bản tin không ghép được về frame gốc", int((j._merge != "both").sum()), (j._merge != "both").sum() == 0),
    ("giá trị nhiệt độ khác", int((b.value_x != b.value_y).sum()), (b.value_x != b.value_y).sum() == 0),
    ("clientId khác", int((b.client_id != b.clientId).sum()), (b.client_id != b.clientId).sum() == 0),
    ("seq (msgid) khác", int((b.msgid != b.seq).sum()), (b.msgid != b.seq).sum() == 0),
    ("lệch thời gian lớn nhất (ms)", round(dt_ms.max(), 3), dt_ms.max() < 1),
    ("bản ghi trùng (ngày, frame)", int(m.duplicated(["day", "frame"]).sum()), m.duplicated(["day", "frame"]).sum() == 0),
    ("nhãn 1 (CSV / gt_labels)", f"{int(lab_csv.label.sum())} / {db.gt_labels.count_documents({})}", int(lab_csv.label.sum()) == db.gt_labels.count_documents({})),
], columns=["kiểm tra", "kết quả", "đạt"])

,kiểm tra,kết quả,đạt
0,số bản tin (gốc / MongoDB),32288 / 32288,True
1,bản tin không ghép được về frame gốc,0,True
2,giá trị nhiệt độ khác,0,True
3,clientId khác,0,True
4,seq (msgid) khác,0,True
5,lệch thời gian lớn nhất (ms),0.999,True
6,"bản ghi trùng (ngày, frame)",0,True
7,nhãn 1 (CSV / gt_labels),544 / 544,True


- Không mất, không thừa, không sai giá trị bản tin nào.
- Thời gian lệch dưới 1 ms vì **BSON lưu ngày giờ đến mili-giây**, còn PCAP lưu đến micro-giây. Với nhịp 300 giây, sai số này không ảnh hưởng: các bộ phát hiện ở mục 4 vẫn khớp pandas tuyệt đối.

### 3.3 Cách mô hình hoá này có chuẩn không?

| Quyết định | Theo nguyên tắc nào |
|---|---|
| Mỗi phép đo một document, thông tin cảm biến trong `meta` | Mẫu khuyến nghị của MongoDB Time Series: `metaField` chứa thông tin **ít thay đổi** dùng để lọc/nhóm (site, InRow, sensor, clientId); MongoDB tự gom các phép đo cùng `meta` vào bucket nén |
| `conn` (đổi mỗi ngày) để **ngoài** `meta` | Giá trị đổi thường xuyên trong `meta` làm vỡ bucket thành nhiều mảnh nhỏ |
| `granularity: "minutes"` cho `telemetry_hist`, `"seconds"` cho `telemetry` | Chọn theo tần suất ghi: dữ liệu gốc 5 phút/bản tin; replay tua ×60 thì 5 giây/bản tin. Với dữ liệu gốc, `"minutes"` cho **152 bucket**, còn `"seconds"` cho 2.716 bucket |
| Hồ sơ cảm biến, tủ, phiên MQTT ở collection riêng, nối bằng `$lookup` khi cần | Dữ liệu tham chiếu ít đổi, đọc nhiều; không nhân bản vào 32.288 document đo |
| Nhãn (`gt_labels`, `gt_incidents`) tách khỏi telemetry | Tránh rò nhãn vào bộ phát hiện (notebook 03, mục 2) |
| JSON Schema validator cho `devices`, `incidents`, `actions` | Chặn dữ liệu sai ngay lúc ghi (đã thử: document có `inrow: 99` bị từ chối) |
| TTL chỉ đặt trên `telemetry` (live) | TTL trên dữ liệu 2019 xoá sạch ngay (mục 3.1) |
| Xuất bằng **Extended JSON** (`{"$date": ...}`) | Định dạng chuẩn của `mongoimport` và driver; giữ đúng kiểu ngày giờ và số |
| Nạp lại bằng cách xoá rồi tạo lại collection | Time Series **không hỗ trợ unique index**, nên không chống trùng bằng index được; giữ nguồn gốc `src.day` + `src.frame` để kiểm tra trùng (bảng trên: 0 trùng) |

**Những gì không đưa vào MongoDB và vì sao:** khoảng 69.000 gói còn lại (ACK thuần, NTP, mở/đóng TCP) chỉ là "giấy tờ" của giao thức, được tóm tắt thành `traffic_daily` và `mqtt_sessions`. File `.xml` trùng hoàn toàn với CSV.

## 4. Chạy bộ phát hiện trên MongoDB và đối chiếu với pandas

Các pipeline ở `src/dad_mongo_pipelines.py` viết lại đúng logic của `src/dad_mining.py`. Chạy trên `telemetry_hist` (chu kỳ gốc P = 300,1 s) và so với kết quả pandas của notebook 02:

In [13]:
packets, pubs = load_all(RAW, CACHE)
pubs = dm.add_baseline(pubs); nodes = dm.node_slots(pubs)
H = db.telemetry_hist
def timed(pipe):
    t = time.time(); r = list(H.aggregate(pipe, allowDiskUse=True)); return r, (time.time() - t) * 1000
r_d1, t1 = timed(mp.d1_silence()); r_d2, t2 = timed(mp.d2_burst()); r_d4, t4 = timed(mp.d4_node()); r_d4s, t4s = timed(mp.d4_sensor())
r_ph, tp = timed(mp.physics()); r_rg, tr = timed(mp.range_check())
s = pubs.sort_values(["client_id", "ts"]).copy(); s["c6"] = (s.dev < -dm.DEV_T).groupby(s.client_id).transform(lambda x: x.rolling(dm.DEV_W).sum())
d4m = pd.DataFrame(r_d4); same_slots = set(zip(d4m.inrow, d4m.slot)) == set(zip(nodes[nodes.neg_count6 >= dm.DEV_K].inrow, nodes[nodes.neg_count6 >= dm.DEV_K].slot))
pd.DataFrame([
    ("D1 im lặng: bản tin đến sau > 1,5P", len(r_d1), int((pubs.gap_s > 1.5 * dm.PERIOD_S).sum()), t1),
    ("D2 gửi dồn: bản tin đến sớm < 0,8P, cùng kết nối", len(r_d2), int(((pubs.gap_s < 0.8 * dm.PERIOD_S) & ~pubs.new_conn).sum()), t2),
    ("D4 theo InRow: slot báo (≥ 3/6)", len(r_d4), int((nodes.neg_count6 >= dm.DEV_K).sum()), t4),
    ("D4 theo cảm biến: bản tin báo", len(r_d4s), int((s.c6 >= dm.DEV_K).sum()), t4s),
    ("Vi phạm TFEU < TFSU < TAS < TAR", len(r_ph), 0, tp),
    ("Vi phạm dải hợp lệ", len(r_rg), 0, tr),
], columns=["bộ phát hiện", "MongoDB", "pandas (notebook 02)", "thời gian chạy (ms)"]).assign(khớp=lambda d: d.MongoDB == d["pandas (notebook 02)"]).round(0)

,bộ phát hiện,MongoDB,pandas (notebook 02),thời gian chạy (ms),khớp
0,"D1 im lặng: bản tin đến sau > 1,5P",64,64,110.0,True
1,"D2 gửi dồn: bản tin đến sớm < 0,8P, cùng kết nối",248,248,79.0,True
2,D4 theo InRow: slot báo (≥ 3/6),85,85,109.0,True
3,D4 theo cảm biến: bản tin báo,497,497,162.0,True
4,Vi phạm TFEU < TFSU < TAS < TAR,0,0,42.0,True
5,Vi phạm dải hợp lệ,0,0,436.0,True


In [14]:
print("D4 theo InRow: tập slot báo động của MongoDB trùng hoàn toàn pandas:", same_slots)
rows = []
for g in db.gt_incidents.find().sort("_id", 1):
    r = list(H.aggregate(mp.investigate(g["inrow"], g["start"], g["end"]), allowDiskUse=True))[0]
    e = g["evidence"]
    mine = {"lost": int(r["lost"]), "extra": int(r["extra"]), "dropped_slots": int(r["dropped_slots"]), "drop_c": None if r.get("drop_c") is None else int(r["drop_c"]), "sensors": int(r["sensors"])}
    ref = {"lost": e["lost_msgs"], "extra": e["extra_msgs"], "dropped_slots": e["dropped_slots"], "drop_c": e["drop_c"], "sensors": e["sensors"]}
    rows.append({"sự cố": g["_id"], "khung giờ": g["local"], **{f"mongo_{k}": v for k, v in mine.items()}, "khớp pandas": mine == ref})
pd.DataFrame(rows)

D4 theo InRow: tập slot báo động của MongoDB trùng hoàn toàn pandas: True


,sự cố,khung giờ,mongo_lost,mongo_extra,mongo_dropped_slots,mongo_drop_c,mongo_sensors,khớp pandas
0,GT-01,Tue 22/10 00:15–00:55,16,0,0,NaN,4,True
1,GT-02,Tue 22/10 01:25–02:25,24,0,0,NaN,4,True
2,GT-03,Wed 23/10 04:10–06:15,0,0,17,2.0,4,True
3,GT-04,Thu 24/10 03:05–03:22,0,56,3,2.0,4,True
4,GT-05,Thu 24/10 23:03–00:00,40,0,0,NaN,4,True
5,GT-06,Fri 25/10 03:25–05:00,60,0,0,NaN,4,True
6,GT-07,Fri 25/10 06:00–06:25,0,96,6,1.0,4,False
7,GT-08,Fri 25/10 14:27–16:22,0,0,15,2.0,4,True
8,GT-09,Sat 26/10 03:00–05:00,40,0,0,NaN,4,True
9,GT-10,Sat 26/10 06:00–06:25,0,96,6,2.0,4,True


- **Mọi bộ phát hiện cho kết quả trùng khớp 100%** với pandas, kể cả tập slot của D4 (dù MongoDB dùng `$median` xấp xỉ). Pipeline điều tra khớp cả 11 sự cố. Mỗi pipeline chạy dưới 1 giây trên cả tuần dữ liệu.
- Như vậy các truy vấn trong `MONITOR_SPEC.md` **đã được chạy thật**, không còn là phác thảo.

### 4.1 Bẫy 2: so sánh với `null` trong MQL

Trong MongoDB, `null` được coi là **nhỏ hơn mọi số**. Bản tin chưa đủ baseline có `dev = null`, nên điều kiện `{$lt: ["$dev", -0.8]}` lại **đúng**:

In [15]:
bad = mp.d4_node()
for st in bad:
    if "$setWindowFields" in st and "negCount" in st["$setWindowFields"]["output"]:
        st["$setWindowFields"]["output"]["negCount"]["$sum"] = {"$cond": [{"$lt": ["$dev", -0.8]}, 1, 0]}
print("D4 không chặn null:", len(list(H.aggregate(bad))), "slot báo | có chặn $isNumber:", len(r_d4), "slot báo")

D4 không chặn null: 176 slot báo | có chặn $isNumber: 85 slot báo


Thiếu một điều kiện `$isNumber` là số cảnh báo tăng hơn gấp đôi, toàn báo giả. Mọi pipeline trong `dad_mongo_pipelines.py` đều chặn trường hợp này.

## 5. Thử kịch bản demo: dời thời gian và tua ×60

`dad_mongo_load.load_scenario()` chép **1 ngày làm nóng** (thứ Hai, để D4 có baseline) và **một kịch bản** từ `telemetry_hist` sang `telemetry`. Thời gian được dời để kết thúc ở *hiện tại* và nén ×60 (5 phút thành 5 giây). Sau đó chạy pipeline với **P = 300,1 / 60 ≈ 5 giây**:

In [16]:
rows = []
gt_key = {"duplication": "^Thu 24/10 03", "modification": "^Wed", "interception": "^Fri 25/10 03"}
for sc in ["duplication", "modification", "interception"]:
    inf = loader.load_scenario(URI, DB, sc, speed=60)
    P = inf["period_s"]; T = db.telemetry; start = inf["scenario_start"]
    g = db.gt_incidents.find_one({"local": {"$regex": gt_key[sc]}})
    t_gt = T.find_one({"orig_ts": {"$gte": g["start"]}}, sort=[("orig_ts", 1)])["ts"]
    d2 = [x for x in T.aggregate(mp.d2_burst(P=P)) if x["ts"] >= start]
    d4 = [x for x in T.aggregate(mp.d4_node(slot_s=P)) if x["slot"] >= start]
    # D1 live: kiểm tra mỗi chu kỳ "now - lần nhận cuối"
    d1_first = None
    if sc == "interception":
        gaps = sorted((x for x in T.aggregate(mp.d1_silence(P=P)) if x["ts"] >= start), key=lambda x: x["ts"])
        last = gaps[0]["ts"] - timedelta(seconds=gaps[0]["gapP"] * P)
        for k in np.arange(1.0, 3.0, 0.1):
            hit = [x for x in T.aggregate(mp.silence_now(last + timedelta(seconds=k * P), P=P)) if x["silentSensors"] > 0]
            if hit: d1_first = (last + timedelta(seconds=k * P) - t_gt).total_seconds(); break
    rows.append({"kịch bản": sc, "documents": inf["docs"], "P (giây)": round(P, 2),
                 "D1 live: số InRow im": len(hit) if sc == "interception" else 0,
                 "D2: bản tin báo": len(d2), "D4: slot báo": len(d4),
                 "giây tới cảnh báo đầu": round(d1_first if sc == "interception" else
                                               ((min(x["ts"] for x in d2) if d2 else min(x["slot"] for x in d4) + timedelta(seconds=P)) - t_gt).total_seconds(), 1)})
pd.DataFrame(rows)

,kịch bản,documents,P (giây),D1 live: số InRow im,D2: bản tin báo,D4: slot báo,giây tới cảnh báo đầu
0,duplication,4796,5.0,0,56,4,1.5
1,modification,5152,5.0,0,0,22,10.6
2,interception,4916,5.0,1,0,0,8.0


- Mỗi kịch bản chỉ bật đúng bộ phát hiện của nó, không bộ nào báo nhầm: chèn gói → D2, sửa giá trị → D4, chặn gói → D1 (chỉ InRow 13).
- Ở kịch bản chèn gói, D4 cũng báo vài slot vì các bản tin chèn mang giá trị thấp (đúng như điều tra ở notebook 03). Số slot D4 có thể chênh ±1 giữa các lần chạy: ranh giới slot (`$dateTrunc`) phụ thuộc thời điểm dữ liệu được dời về.
- Thời gian tới cảnh báo đầu (đo từ lúc sự cố bắt đầu, ở tốc độ ×60): D2 khoảng 1–2 giây, D4 khoảng 10–15 giây, D1 khoảng 7–8 giây với kiểm tra live `silence_now`. Khớp dự tính ở notebook 03, mục 6.
- Riêng D1: pipeline `d1_silence` trên dữ liệu đã lưu chỉ thấy khoảng hở **khi bản tin kế tiếp tới** (muộn). Khi chạy live phải dùng `silence_now(now)` mỗi chu kỳ.

## 6. Nạp lên MongoDB Atlas

```bash
pip install -r requirements.txt
python src/dad_export.py                                   # tạo lại data/mongo/*.ndjson.gz từ PCAP (đã có sẵn trong repo)
MONGODB_URI="mongodb+srv://<user>:<pass>@<cluster>/" python src/dad_mongo_load.py --db dad
```
Hoặc dùng `mongoimport` cho từng file (Time Series cần tạo collection trước bằng script, vì `mongoimport` không tạo được Time Series):
```bash
mongoimport --uri "$MONGODB_URI/dad" --collection devices --file <(zcat data/mongo/devices.ndjson.gz)
```
**Không** đưa chuỗi kết nối hay mật khẩu vào repo hoặc video demo (thể lệ cuộc thi, mục 6.1).

## 7. Tóm tắt

| Đã có | Ở đâu |
|---|---|
| 9 collection dựng từ dataset gốc, Extended JSON, ~490 KB | `data/mongo/*.ndjson.gz`, `data/mongo/manifest.csv` |
| Script xuất từ PCAP/CSV | `src/dad_export.py` |
| Script nạp: Time Series, TTL, validator, index, kịch bản demo dời thời gian | `src/dad_mongo_load.py` |
| Bộ phát hiện + điều tra bằng Aggregation Pipeline, **đã chạy thật và khớp pandas** | `src/dad_mongo_pipelines.py` |
| Kiểm tra toàn vẹn từng bản tin: 0 mất, 0 sai giá trị, lệch thời gian < 1 ms | mục 3.2 |
| Cách mô hình hoá theo nguyên tắc của MongoDB Time Series | mục 3.3 |
| Hai bẫy đã kiểm chứng: TTL xoá dữ liệu 2019; `null < số` trong MQL | mục 3.1, 4.1 |